# RAGspector Base Model

---

## Building In-Built RAG

---

**Installing Dependencies**

In [2]:
!pip install langchain-community jq

INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 1.2 MB/s eta 0:00:02
   ------------- -------------------------- 0.8/2.4 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aext-assistant-server 4.1.0 requires anaconda-cloud-auth>=0.7.1, which is not installed.


In [3]:
!pip install chromadb

  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/23.5 MB 2.0 MB/s eta 0:00:12
   -- ------------------------------------- 1.3/23.5 MB 2.1 MB/s eta 0:00:11
   --- ------------------------------------ 1.8/23.5 MB 2.2 MB/s eta 0:00:10
   ---- ----------------------------------- 2.4/23.5 MB 2.3 MB/s eta 0:00:10
   ---- ----------------------------------- 2.6/23.5 MB 2.3 MB/s eta 0:00:10
   ----- ---------------------------------- 3.4/23.5 MB 2.3 MB/s eta 0:00:09
   ------ --------------------------------- 3.9/23.5 MB 2.4 MB/s eta 0:00:09
   -------- ------------------------------- 4.7/23.5 MB 2.5 MB/s eta 0:00:08
   --------- ------------------------------ 5.5/23.5 MB 2.7 MB/s eta 0:00:07
   ---------- ----------------------------- 6.3/23.5 MB 2.8 MB/s eta 0:00:07
   ----------- ---------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
botocore 1.34.69 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 7.36.2 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.36.2 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 7.36.2 which is incompatible.


**Importing Required Packages**

In [4]:
import gzip
import json
from pathlib import Path

from datasets import load_dataset
from tqdm import tqdm

d:\UNI\Sem_7\FYP2023\WorkSpace\Implementation_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**CONFIG**

In [5]:
# dataset configuration
DATASET_NAME = "wikimedia/wikipedia"
DATASET_CONFIG = "20231101.en"
SPLIT = "train"

# Output location
OUTPUT_DIR = Path("../content/RAGspector/data/raw")
OUTPUT_FILE = OUTPUT_DIR / "wikipedia_articles.jsonl.gz"
MAX_ARTICLES = 10000

**Load wikipedia articles**

In [4]:
# Streaming wikipedia articles

OUTPUT_DIR.mkdir(parents = True, exist_ok = True)

dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split = SPLIT, streaming = True)

In [5]:
# saving each article as JSON object

article_count = 0
with gzip.open(OUTPUT_FILE, mode = 'wt', encoding = "utf-8") as output_file:
    for article in tqdm(dataset, desc = "Downloading articles"):
        if article_count >= MAX_ARTICLES:
            break

        article_id = str(article["id"])
        title = article["title"].strip()
        url = article["url"].strip()
        text = article["text"].strip()

        # skip empty or unusable records
        if not title or not text:
            continue

        record = {
            "id" : article_id,
            "title" : title,
            "url" : url,
            "text" : text
        }

        output_file.write(json.dumps(record, ensure_ascii = False) + "\n")
        article_count += 1
print(f"Saved to: {OUTPUT_FILE}")

Saved to: ..\content\RAGspector\data\raw\wikipedia_articles.jsonl.gz


In [6]:
# Extracting the compressed JSONL file
import shutil

INPUT_FILE = Path("../content/RAGspector/data/raw/wikipedia_articles.jsonl.gz")
OUTPUT_FILE = Path("../content/RAGspector/data/raw/wikipedia_articles.jsonl")

with gzip.open(INPUT_FILE, "r") as compressed_file:
    with open(OUTPUT_FILE, "wb") as extracted_file:
        shutil.copyfileobj(compressed_file, extracted_file)

print(f"Extraction complete: {OUTPUT_FILE}")

Extraction complete: ..\content\RAGspector\data\raw\wikipedia_articles.jsonl


In [7]:
# Loading the articles into Document data structure using JSONLoader
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path = "../content/RAGspector/data/raw/wikipedia_articles.jsonl",
    jq_schema = ".",
    content_key = "text",
    metadata_func = lambda record, metadata: {
        **metadata,
        "article_id": record["id"],
        "title": record["title"],
        "url": record["url"]
    },
    json_lines = True
)

docs = loader.load()

C:\Users\aksha\AppData\Local\Temp\ipykernel_18564\563728105.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


In [8]:
docs[0].metadata

{'source': 'D:\\UNI\\Sem_7\\FYP2023\\WorkSpace\\Implementation_2\\content\\RAGspector\\data\\raw\\wikipedia_articles.jsonl',
 'seq_num': 1,
 'article_id': '12',
 'title': 'Anarchism',
 'url': 'https://en.wikipedia.org/wiki/Anarchism'}

**Chunk documents**

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 200,
    length_function = len,
    separators = ["\n\n", "\n", ". ", " ", ""])
all_splits = text_splitter.split_documents(docs)
print("Number of chunks: ", len(all_splits))

Number of chunks:  84511


**Generate Embedding**

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

texts = [chunk.page_content for chunk in all_splits]
sen_trans = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model dimensions: ", sen_trans.get_embedding_dimension())

# Generating embeddings
embeddings = sen_trans.encode(texts, show_progress_bar= True, normalize_embeddings= True)
print("Embedding shape: ", embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3214.38it/s]


Embedding model dimensions:  384


Batches: 100%|██████████| 2641/2641 [51:31<00:00,  1.17s/it]  


Embedding shape:  (84511, 384)


**Store Embeddings in ChromaDB**

In [11]:
# Importing required packages
import os
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid

# Configurations

collection_name = "Wikipedia_20231101_10k"
persist_directory = "../content/RAGspector/VectorStore"


# Create chromadb client and collection
os.makedirs(persist_directory, exist_ok = True)
client = chromadb.PersistentClient(path = persist_directory)

# Get or create collection
collection = client.get_or_create_collection(name = collection_name, metadata = {"Description": "HTML document embeddings for RAG"})
print("Exisiting documents in the collection: ", collection.count())

Exisiting documents in the collection:  0


In [12]:
# Prepare data for chromadb
ids = []
metadatas = []
chunk_text = []
embeddings_list = []

for i, (chunk, embedding) in enumerate(zip(all_splits, embeddings)):
    # Generate unique id:
    doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
    ids.append(doc_id)

    # Prepare metadata
    metadata = dict(chunk.metadata)
    metadata["doc_index"] = i
    metadata["content_length"] = len(chunk.page_content)
    metadatas.append(metadata)

    # append chunk content
    chunk_text.append(chunk.page_content)

    # append the corresponding embedding
    embeddings_list.append(embedding.tolist())


# --- CHUNKING & ADDING IN BATCHES ---
MAX_BATCH_SIZE = 5000  # Keeping it safely under the 5,461 limit

for i in range(0, len(ids), MAX_BATCH_SIZE):
    batch_ids = ids[i : i + MAX_BATCH_SIZE]
    batch_embeddings = embeddings_list[i : i + MAX_BATCH_SIZE]
    batch_metadatas = metadatas[i : i + MAX_BATCH_SIZE]
    batch_documents = chunk_text[i : i + MAX_BATCH_SIZE]

    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        metadatas=batch_metadatas,
        documents=batch_documents,
    )
    print(
        f"Uploaded batch {i // MAX_BATCH_SIZE + 1}: Chunks {i} to {i + len(batch_ids)}"
    )


print(f"\nSuccessfully added {len(all_splits)} chunks to the vector store")
print(f"Total chunks in the collection: {collection.count()}")


Uploaded batch 1: Chunks 0 to 5000
Uploaded batch 2: Chunks 5000 to 10000
Uploaded batch 3: Chunks 10000 to 15000
Uploaded batch 4: Chunks 15000 to 20000
Uploaded batch 5: Chunks 20000 to 25000
Uploaded batch 6: Chunks 25000 to 30000
Uploaded batch 7: Chunks 30000 to 35000
Uploaded batch 8: Chunks 35000 to 40000
Uploaded batch 9: Chunks 40000 to 45000
Uploaded batch 10: Chunks 45000 to 50000
Uploaded batch 11: Chunks 50000 to 55000
Uploaded batch 12: Chunks 55000 to 60000
Uploaded batch 13: Chunks 60000 to 65000
Uploaded batch 14: Chunks 65000 to 70000
Uploaded batch 15: Chunks 70000 to 75000
Uploaded batch 16: Chunks 75000 to 80000
Uploaded batch 17: Chunks 80000 to 84511

Successfully added 84511 chunks to the vector store
Total chunks in the collection: 84511


### Need to convert the below code into functions

**RAG Retriever**

In [ ]:
"""
"""
This pipeline handles the query-based retrival from the vector-store

Required components

1. Vector Store: collection
2. Embedding generator: sen_trans

"""

# Retrieving Relevant documents for the query

query = "how can a nature of an appeal can vary?"
top_k = 5
# score_threshold = 0.0

# generating query embedding:
query_embedding = sen_trans.encode(query, normalize_embeddings=True)


# searching the vector store for relevant records
results = collection.query(query_embeddings= [query_embedding.tolist()], n_results= top_k)

if results['documents'] and results['documents'][0]:
    documents = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]
    ids = results['ids'][0]

retrieved_docs = []

# calculating the similarity score for each retrived documents
for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
    similarity_score = 1 - distance

    retrieved_docs.append({
        'id': doc_id,
        'document' : document,
        'metadata' : metadata,
        'similarity_score': similarity_score,
        'distance' : distance,
        'rank' : i + 1
    })

print(f"Number of documents retrieved after searching {len(retrieved_docs)}")

"""

In [ ]:
retrieved_docs

In [ ]:
docs[4].page_content

**Augmented Generation Pipeline**

In [ ]:
"""# Importing Reuired Libraries
import os
from dotenv import load_dotenv
from openai import OpenAI

# loading the environment variables
load_dotenv()

# get the environment variable from .env file
Llama_3_1_8B_instruct_api_key = "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"

# Create a client
llm_client = OpenAI(
    api_key= Llama_3_1_8B_instruct_api_key,
    base_url = "https://openrouter.ai/api/v1"
)

# Building the context
context = "\n\n".join([doc['document'] for doc in retrieved_docs]) if retrieved_docs else ""

# Building the prompt
prompt = f"""Use the following context to answer the question concisely.
            context : {context}

            query: {query}

            Answer: """

# Getting the response from the model
response = llm_client.chat.completions.create(
    model = "nvidia/nemotron-3-ultra-550b-a55b:free",
    messages = [{
        "role" : "user",
        "content" : prompt
    }],
    temperature= 0.1,
    max_tokens= 1024
)
"""

In [ ]:
"""answer = response.choices[0].message.content
print(answer)"""

## C2: Building Test Generator

---

**CONFIG**

In [13]:
DATASET_NAME = "hotpotqa/hotpot_qa"
DATASET_CONFIG = "distractor"
DATASET_SPLIT = "validation"

BENCHMARK_SIZE = 100
RANDOM_SEED = 13

BENCHMARK_DIR = Path("../content/RAGspector/data/benchmark")
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_FILE = BENCHMARK_DIR / "hotpotqa_subset.jsonl"

**Load HotpotQA Benchmark Dataset**

In [14]:
dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=DATASET_SPLIT
)

print("Total validation examples:", len(dataset))

dataset = dataset.shuffle(seed=RANDOM_SEED)

benchmark_subset = dataset.select(
    range(min(BENCHMARK_SIZE, len(dataset)))
)

print("Selected benchmark examples:", len(benchmark_subset))

d:\UNI\Sem_7\FYP2023\WorkSpace\Implementation_2\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aksha\.cache\huggingface\hub\datasets--hotpotqa--hotpot_qa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\UNI\Sem_7\FYP2023\WorkSpace\Implementation_2\.venv\Lib\site-packages\huggingface_hub\file_d

Total validation examples: 7405
Selected benchmark examples: 100


In [15]:
benchmark_subset

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 100
})

In [16]:
# converting records into benchmark structure
def convert_hotpotqa_record(record):

    context_titles = record["context"]["title"]
    context_sentences = record["context"]["sentences"]

    context_documents = []

    for title, sentences in zip(
        context_titles,
        context_sentences
    ):
        context_documents.append({
            "title": title,
            "text": " ".join(sentences)
        })

    supporting_facts = record["supporting_facts"]

    gold_evidence = []

    for title, sent_id in zip(
        supporting_facts["title"],
        supporting_facts["sent_id"]
    ):

        if title not in context_titles:
            continue

        title_index = context_titles.index(title)
        sentences = context_sentences[title_index]

        if 0 <= sent_id < len(sentences):

            gold_evidence.append({
                "title": title,
                "sentence_id": sent_id,
                "text": sentences[sent_id]
            })

    return {
        "case_id": str(record["id"]),
        "query": record["question"],
        "reference_answer": record["answer"],

        "gold_evidence": gold_evidence,

        "corpus_document_ids": list(
            dict.fromkeys(context_titles)
        ),

        "task_type": record["type"],
        "difficulty": record["level"],

        "context_documents": context_documents
    }

In [17]:
# create and store benchmark
benchmark_cases = [
    convert_hotpotqa_record(record)
    for record in tqdm(benchmark_subset)
]

with open(BENCHMARK_FILE, "w", encoding="utf-8") as f:

    for case in benchmark_cases:
        f.write(json.dumps(case, ensure_ascii=False) + "\n")

print("Benchmark saved:", BENCHMARK_FILE)
print("Total cases:", len(benchmark_cases))

100%|██████████| 100/100 [00:00<00:00, 6245.15it/s]

Benchmark saved: ..\content\RAGspector\data\benchmark\hotpotqa_subset.jsonl
Total cases: 100


In [18]:
benchmark_cases[44]

{'case_id': '5ac083b0554299294b21900d',
 'query': "What kind of beverage does Planter's Punch and Black Velvet have in common?",
 'reference_answer': 'cocktail',
 'gold_evidence': [{'title': "Planter's Punch",
   'sentence_id': 0,
   'text': "Planter's Punch is an IBA Official Cocktail made of dark rum, several juices, grenadine syrup, sugar syrup, and Angostura bitters."},
  {'title': 'Black Velvet (beer cocktail)',
   'sentence_id': 0,
   'text': 'The Black Velvet is a beer cocktail made from stout (often Guinness) and white, sparkling wine, traditionally Champagne.'}],
 'corpus_document_ids': ['Black Velvet Flag',
  "Planter's Punch",
  'Velvet painting',
  'Black rose (symbolism)',
  'Edgar Leeteg',
  'Velvet Elvis',
  'Black Velvet (beer cocktail)',
  'Bigger and Blackerer',
  'Black Velvet (magazine)',
  'Velvet (bus company)'],
 'task_type': 'comparison',
 'difficulty': 'hard',
 'context_documents': [{'title': 'Black Velvet Flag',
   'text': 'Black Velvet Flag was a New York Cit

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [20]:
# defining a function for retrival and generation

import time

# get the environment variable from .env file
Llama_3_1_8B_instruct_api_key = "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"

# Create a client
llm_client = OpenAI(
    api_key= Llama_3_1_8B_instruct_api_key,
    base_url = "https://openrouter.ai/api/v1"
)

RAG_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"


def run_rag(
    query,
    top_k=5,
    context_override=None
):
    """
    Execute the in-built RAG pipeline.

    Parameters
    ----------
    query : str
        User query.

    top_k : int
        Number of documents to retrieve.

    context_override : list[dict] or None
        If provided, use these documents directly as context
        instead of retrieving documents from ChromaDB.

    Returns
    -------
    dict
        Structured RAG result.
    """

    start_time = time.perf_counter()

    # --------------------------------------------------
    # 1. RETRIEVAL
    # --------------------------------------------------

    if context_override is None:

        query_embedding = sen_trans.encode(
            query,
            normalize_embeddings=True
        )

        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )

        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]
        ids = results["ids"][0]

        retrieved_docs = []

        for rank, (doc_id, document, metadata, distance) in enumerate(
            zip(ids, documents, metadatas, distances),
            start=1
        ):

            retrieved_docs.append({
                "id": doc_id,
                "document": document,
                "metadata": metadata,
                "distance": float(distance),
                "rank": rank
            })

        retrieval_performed = True

    else:

        # Use the supplied context instead of retrieving.
        retrieved_docs = context_override

        retrieval_performed = False

    # --------------------------------------------------
    # 2. CONTEXT CONSTRUCTION
    # --------------------------------------------------

    context = "\n\n".join(
        doc["document"]
        for doc in retrieved_docs
    )

    # --------------------------------------------------
    # 3. PROMPT CONSTRUCTION
    # --------------------------------------------------

    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    # --------------------------------------------------
    # 4. GENERATION
    # --------------------------------------------------

    response = llm_client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=1024
    )

    answer = response.choices[0].message.content

    # --------------------------------------------------
    # 5. RESULT CONSTRUCTION
    # --------------------------------------------------

    latency = time.perf_counter() - start_time

    effective_context_ids = [
        doc["id"]
        for doc in retrieved_docs
    ]

    usage = getattr(response, "usage", None)

    token_usage = None

    if usage is not None:
        token_usage = {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens
        }

    return {

        "query": query,

        "answer": answer,

        "retrieved_documents": retrieved_docs,

        "latency_seconds": latency,

        "token_usage": token_usage,

        "metadata": {
            "effective_query": query,
            "effective_context_ids": effective_context_ids,
            "applied_retrieval_options": (
                {"top_k": top_k}
                if retrieval_performed else {}
            ),
            "retrieval_performed": retrieval_performed,
            "context_override_used": context_override is not None
        }
    }

In [ ]:
# testing run_rag function
test_query = "How can the nature of an appeal vary?"

test_result = run_rag(
    query=test_query,
    top_k=5
)

print("Question:", test_result["query"])
print("\nAnswer:", test_result["answer"])
print("\nLatency:", test_result["latency_seconds"])

print("\nRetrieved documents:")

for doc in test_result["retrieved_documents"]:
    print(
        doc["rank"],
        doc["id"],
        doc["metadata"].get("title")
    )

before proceeding further, I ran a overlap check to determine how effectively does the benchmark cover our KB. I found out that it only covers 4% of the KB.


---

## Building a custom benchmark

---

In [ ]:
# Extracting (Inspecting) source passages from the vectorDB and inspecting metadata

sample = collection.get(
    limit=5,
    include=["documents", "metadatas"]
)

print("Number of records:", len(sample["ids"]))

for i in range(len(sample["ids"])):
    print("\nChunk ID:", sample["ids"][i])
    print("Metadata:", sample["metadatas"][i])
    print("Document:", sample["documents"][i][:300])

Number of records: 5

Chunk ID: doc_ac84e3cf_0
Metadata: {'doc_index': 0, 'source': 'D:\\UNI\\Sem_7\\FYP2023\\WorkSpace\\Implementation_2\\content\\RAGspector\\data\\raw\\wikipedia_articles.jsonl', 'article_id': '12', 'content_length': 572, 'title': 'Anarchism', 'seq_num': 1, 'url': 'https://en.wikipedia.org/wiki/Anarchism'}
Document: Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state 

Chunk ID: doc_bf8fcd38_1
Metadata: {'content_length': 1458, 'doc_index': 1, 'article_id': '12', 'url': 'https://en.wikipedia.org/wiki/Anarchism', 'title': 'Anarchism', 'seq_num': 1, 'source': 'D:\\UNI\\Sem_7\\FYP2023\\WorkSpace\\Implementation_2\\content\\RAGspector\\data\\raw\\wikipedia_articles.jsonl'}
Document: Humans have lived in societies without formal

In [31]:
# build an atricle-to-chunk mapping

from collections import defaultdict

BATCH_SIZE = 500

total_chunks = collection.count()

article_chunks = defaultdict(list)

for offset in range(0, total_chunks, BATCH_SIZE):

    batch = collection.get(
        limit=BATCH_SIZE,
        offset=offset,
        include=["metadatas"]
    )

    for chunk_id, metadata in zip(
        batch["ids"],
        batch["metadatas"]
    ):

        if metadata and metadata.get("title"):

            title = metadata["title"]

            article_chunks[title].append(chunk_id)


print("Total chunks:", total_chunks)
print("Unique articles:", len(article_chunks))

Total chunks: 84511
Unique articles: 10000


In [37]:
# Sampling different articles for question generation

import random

RANDOM_SEED = 42

random.seed(RANDOM_SEED)

TARGET_ARTICLES = 40

available_titles = list(article_chunks.keys())

selected_titles = random.sample(
    available_titles,
    min(TARGET_ARTICLES, len(available_titles))
)

selected_chunk_ids = []

for title in selected_titles:

    chunk_ids = article_chunks[title]

    selected_chunk_ids.append(
        random.choice(chunk_ids)
    )

print("Selected articles:", len(selected_titles))

Selected articles: 40


In [39]:
# Retrieving the source passage

selected_records = collection.get(
    ids=selected_chunk_ids,
    include=["documents", "metadatas"]
)

source_passages = []

for chunk_id, document, metadata in zip(
    selected_records["ids"],
    selected_records["documents"],
    selected_records["metadatas"]
):

    if not document:
        continue

    if len(document.strip()) < 300:
        continue

    if len(document) > 3000:
        continue

    source_passages.append({
        "source_id": chunk_id,
        "title": metadata.get("title"),
        "text": document.strip()
    })


print("Eligible source passages:", len(source_passages))

Eligible source passages: 39


In [ ]:
# inspect three passages to ensure that the extraction has preserved meaningful content.

for i, passage in enumerate(source_passages[:3], start=1):

    print(f"\n{'=' * 70}")
    print(f"PASSAGE {i}")
    print("Article:", passage["title"])
    print("Chunk ID:", passage["source_id"])
    print("Character count:", len(passage["text"]))
    print("\nText:\n", passage["text"])

In [42]:
# function for question generation using sampled passages using LLM

import json
import math

BENCHMARK_GENERATOR_MODEL = RAG_MODEL

def generate_questions_batch(passages):

    input_passages = [
        {
            "source_id": p["source_id"],
            "title": p["title"],
            "passage": p["text"]
        }
        for p in passages
    ]

    prompt = f"""
You are generating benchmark questions for evaluating
a Retrieval-Augmented Generation (RAG) system.

Generate exactly ONE high-quality question for each
provided source passage.

STRICT REQUIREMENTS:

1. The question must be answerable using ONLY its
   corresponding passage.
2. The reference answer must be factually supported
   by that passage.
3. Do not introduce external knowledge.
4. Do not include the answer in the question.
5. Avoid vague questions and questions with ambiguous answers.
6. The evidence_quote must be copied exactly from the passage.
7. Prefer questions that test understanding rather than
   simple copying of a sentence.
8. If a passage does not support a meaningful question,
   mark its status as "skip".

For every passage, return:

- source_id
- status: "ok" or "skip"
- question
- answer
- evidence_quote
- task_type: "factual", "conceptual", or "descriptive"
- difficulty: "easy", "medium", or "hard"

Return ONLY a valid JSON array.

Input passages:

{json.dumps(input_passages, ensure_ascii=False)}
"""

    response = llm_client.chat.completions.create(
        model=BENCHMARK_GENERATOR_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        max_tokens=2500
    )

    raw_output = response.choices[0].message.content.strip()

    # Remove Markdown code fences if the model adds them
    if raw_output.startswith("```"):
        raw_output = raw_output.split("\n", 1)[1]
        raw_output = raw_output.rsplit("```", 1)[0]

    generated = json.loads(raw_output)

    return generated

In [41]:
# selecting random 20 passages from the sampled passages

import random

random.seed(42)

generation_sources = random.sample(
    source_passages,
    20
)

print("Selected passages:", len(generation_sources))
print(
    "Unique articles:",
    len({p["title"] for p in generation_sources})
)

Selected passages: 20
Unique articles: 20


In [43]:
BATCH_SIZE = 4
PILOT_SIZE = 20

# generation_sources = source_passages[:PILOT_SIZE]

generated_candidates = []

for start in range(0, len(generation_sources), BATCH_SIZE):

    batch = generation_sources[
        start:start + BATCH_SIZE
    ]

    print(
        f"Generating questions for passages "
        f"{start + 1} to {start + len(batch)}"
    )

    generated = generate_questions_batch(batch)

    generated_candidates.extend(generated)


print("Generated candidate records:", len(generated_candidates))

Generating questions for passages 1 to 4
Generating questions for passages 5 to 8
Generating questions for passages 9 to 12
Generating questions for passages 13 to 16
Generating questions for passages 17 to 20
Generated candidate records: 20


In [45]:
# function for validating generated benchmark questions

import re
import hashlib

def normalize_text(text):

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)

    return text


source_lookup = {
    p["source_id"]: p
    for p in generation_sources
}

seen_questions = set()


def validate_candidate(candidate):

    errors = []

    source_id = candidate.get("source_id")

    # 1. Source existence
    source = source_lookup.get(source_id)

    if source is None:
        errors.append("unknown_source_id")
        return errors

    question = candidate.get("question", "").strip()
    answer = candidate.get("answer", "").strip()
    evidence_quote = candidate.get("evidence_quote", "").strip()

    # 2. Required fields
    if not question:
        errors.append("missing_question")

    if not answer:
        errors.append("missing_answer")

    if not evidence_quote:
        errors.append("missing_evidence_quote")

    if not question or not answer or not evidence_quote:
        return errors

    # 3. Evidence quote verification
    source_text = source["text"]

    normalized_source = normalize_text(source_text)
    normalized_quote = normalize_text(evidence_quote)

    if normalized_quote not in normalized_source:
        errors.append("evidence_quote_not_found")

    # 4. Answer leakage check
    normalized_question = normalize_text(question)
    normalized_answer = normalize_text(answer)

    if normalized_answer in normalized_question:
        errors.append("possible_answer_leakage")

    # 5. Duplicate question check
    question_fingerprint = normalized_question

    if question_fingerprint in seen_questions:
        errors.append("duplicate_question")

    # 6. Label validation
    valid_task_types = {
        "factual", "conceptual", "descriptive"
    }

    valid_difficulties = {
        "easy", "medium", "hard"
    }

    if candidate.get("task_type") not in valid_task_types:
        errors.append("invalid_task_type")

    if candidate.get("difficulty") not in valid_difficulties:
        errors.append("invalid_difficulty")

    return errors

In [46]:
validation_results = []

for candidate in generated_candidates:

    if candidate.get("status") != "ok":
        validation_results.append({
            "candidate": candidate,
            "valid": False,
            "errors": ["generation_skipped"]
        })
        continue

    errors = validate_candidate(candidate)

    if not errors:
        seen_questions.add(
            normalize_text(candidate["question"])
        )

    validation_results.append({
        "candidate": candidate,
        "valid": len(errors) == 0,
        "errors": errors
    })


valid_candidates = [
    r["candidate"]
    for r in validation_results
    if r["valid"]
]

invalid_candidates = [
    r for r in validation_results
    if not r["valid"]
]

print("Total candidates:", len(generated_candidates))
print("Valid candidates:", len(valid_candidates))
print("Invalid candidates:", len(invalid_candidates))

Total candidates: 20
Valid candidates: 18
Invalid candidates: 2


In [ ]:
for i, candidate in enumerate(valid_candidates, start=1):

    source = source_lookup[candidate["source_id"]]

    print(f"\n{'=' * 70}")
    print(f"Candidate: {i}")
    print(f"Article: {source['title']}")
    print(f"Question: {candidate['question']}")
    print(f"Reference answer: {candidate['answer']}")
    print(f"Evidence quote: {candidate['evidence_quote']}")
    print(f"Difficulty: {candidate['difficulty']}")
    print(f"Task type: {candidate['task_type']}")
    print(f"Source passage:\n{source['text']}")

In [48]:
# Converting the candidates into our benchmark schema

import uuid

benchmark_cases_custom = []

# for candidate in approved_candidates:
for candidate in valid_candidates:

    source = source_lookup[candidate["source_id"]]

    case_id = "wiki_" + hashlib.sha1(
        candidate["source_id"].encode()
    ).hexdigest()[:12]

    benchmark_case = {

        "case_id": case_id,

        "query": candidate["question"],

        "reference_answer": candidate["answer"],

        "gold_evidence": [
            {
                "document_id": candidate["source_id"],
                "title": source["title"],
                "text": source["text"],
                "supporting_quote": candidate["evidence_quote"]
            }
        ],

        "gold_evidence_ids": [
            candidate["source_id"]
        ],

        "corpus_document_ids": [
            candidate["source_id"]
        ],

        "task_type": candidate["task_type"],

        "difficulty": candidate["difficulty"],

        "metadata": {
            "source": "Wikipedia",
            "snapshot": "2023-11-01",
            "generation_model": BENCHMARK_GENERATOR_MODEL,
            "generation_method": "LLM-assisted",
            "question_type": "single_hop",
            "human_reviewed": True
        }
    }

    benchmark_cases_custom.append(benchmark_case)


print("Approved benchmark cases:", len(benchmark_cases_custom))

Approved benchmark cases: 18


In [52]:
# saving the benchmarck questions

from pathlib import Path
import json

BENCHMARK_DIR = Path("../content/RAGspector/data/benchmark")

"""BENCHMARK_DIR = (
    Path.cwd() / "data" / "benchmark"
)"""

BENCHMARK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BENCHMARK_FILE = (
    BENCHMARK_DIR / "wikipedia_custom_benchmark_v1.jsonl"
)

with open(BENCHMARK_FILE, "w", encoding="utf-8") as f:

    for case in benchmark_cases_custom:

        f.write(
            json.dumps(case, ensure_ascii=False) + "\n"
        )


print("Benchmark saved at:", BENCHMARK_FILE.resolve())

Benchmark saved at: D:\UNI\Sem_7\FYP2023\WorkSpace\Implementation_2\content\RAGspector\data\benchmark\wikipedia_custom_benchmark_v1.jsonl


In [53]:
# verifying benchmark against vectorDB

benchmark_evidence_ids = {
    evidence_id
    for case in benchmark_cases_custom
    for evidence_id in case["gold_evidence_ids"]
}

verified_ids = set()

for start in range(
    0, len(benchmark_evidence_ids), 100
):

    batch_ids = list(benchmark_evidence_ids)[
        start:start + 100
    ]

    result = collection.get(
        ids=batch_ids,
        include=[]
    )

    verified_ids.update(result["ids"])


print("Total gold evidence IDs:", len(benchmark_evidence_ids))
print("Verified evidence IDs:", len(verified_ids))
print(
    "Evidence availability:",
    f"{len(verified_ids) / len(benchmark_evidence_ids) * 100:.2f}%"
)

Total gold evidence IDs: 18
Verified evidence IDs: 18
Evidence availability: 100.00%


**Establish Baseline Response**

In [54]:
BASELINE_DIR = Path("../content/RAGspector/data/baselines")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_FILE = BASELINE_DIR / "hotpotqa_baselines.jsonl"

In [56]:
def run_baseline_experiment(
    benchmark_cases,
    output_file=BASELINE_FILE,
    top_k=5
):

    baseline_results = []

    with open(output_file, "w", encoding="utf-8") as f:

        for case in tqdm(
            benchmark_cases,
            desc="Running baseline RAG"
        ):

            result = run_rag(
                query=case["query"],
                top_k=top_k
            )

            record = {
                "case_id": case["case_id"],
                "baseline": result
            }

            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

            baseline_results.append(record)

    print("Baseline experiment completed.")
    print("Total cases:", len(baseline_results))
    print("Saved to:", output_file)

    return baseline_results

In [57]:
# Load Benchmark dataset

import json
from pathlib import Path

BENCHMARK_FILE = Path("../content/RAGspector/data/benchmark/wikipedia_custom_benchmark_v1.jsonl")

benchmark_cases = []

with open(BENCHMARK_FILE, "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            benchmark_cases.append(json.loads(line))

print("Total benchmark cases:", len(benchmark_cases))
print("\nSample benchmark case:")
print(json.dumps(benchmark_cases[0], indent=4, ensure_ascii=False))

Total benchmark cases: 18

Sample benchmark case:
{
    "case_id": "wiki_d9430c285947",
    "query": "Why can amides participate in hydrogen bonding with water?",
    "reference_answer": "Amides can participate in hydrogen bonding with water because the carbonyl oxygen acts as a hydrogen bond acceptor and the N–H hydrogen atoms (in primary and secondary amides) act as hydrogen bond donors.",
    "gold_evidence": [
        {
            "document_id": "doc_904c0e24_9793",
            "title": "Amide",
            "text": "Hydrogen bonding and solubility\nBecause of the greater electronegativity of oxygen, the carbonyl (C=O) is a stronger dipole than the N–C dipole. The presence of a C=O dipole and, to a lesser extent a N–C dipole, allows amides to act as H-bond acceptors. In primary and secondary amides, the presence of N–H dipoles allows amides to function as H-bond donors as well. Thus amides can participate in hydrogen bonding with water and other protic solvents; the oxygen atom can

In [58]:
# Retrieval function for benchmark questions

TOP_K = 5  

def retrieve_documents(query, top_k=TOP_K):

    # Generate query embedding
    query_embedding = sen_trans.encode(query).tolist()

    # Retrieve relevant chunks from ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_documents = []

    for i in range(len(results["ids"][0])):

        retrieved_documents.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved_documents

In [59]:
# answer generation function

def generate_rag_answer(query, retrieved_documents):

    context = ""

    for i, document in enumerate(retrieved_documents, start=1):

        context += f"""
        Document {i}
        Title: {document["metadata"].get("title", "Unknown")}
        Content:
        {document["text"]}
        """

    system_prompt = """Use the following context to answer the question concisely."""
    #system_prompt = """
    #You are a question-answering assistant.

    #Answer the user's question using only the information
    #available in the provided context.

    #If the context does not contain sufficient information
    #to answer the question, explicitly state that.

    #Do not introduce unsupported information.
    #"""

    user_prompt = f"""
    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm_client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    answer = response.choices[0].message.content

    usage = None
    if response.usage:
        if hasattr(response.usage, "model_dump"):
            usage = response.usage.model_dump()
        else:
            usage = str(response.usage)

    return answer, usage

In [60]:
# function for calculating retreival metrics

def calculate_retrieval_metrics(retrieved_ids, gold_ids):

    retrieved_ids = retrieved_ids or []
    gold_ids = set(gold_ids or [])

    if not gold_ids:
        return {
            "recall_at_k": None,
            "precision_at_k": None,
            "mrr": None
        }

    retrieved_set = set(retrieved_ids)

    relevant_retrieved = retrieved_set.intersection(gold_ids)

    recall = len(relevant_retrieved) / len(gold_ids)

    precision = (
        len(relevant_retrieved) / len(retrieved_ids)
        if retrieved_ids else 0.0
    )

    reciprocal_rank = 0.0

    for rank, chunk_id in enumerate(retrieved_ids, start=1):

        if chunk_id in gold_ids:
            reciprocal_rank = 1 / rank
            break

    return {
        "recall_at_k": recall,
        "precision_at_k": precision,
        "mrr": reciprocal_rank
    }

In [61]:
# function for executing the RAG on benchmark questions

import time

def run_baseline(case):

    query = case["query"]
    gold_ids = case.get("gold_evidence_ids", [])

    start_time = time.perf_counter()

    # Step 1: Retrieve documents
    retrieved_documents = retrieve_documents(query)

    # Step 2: Generate answer
    answer, usage = generate_rag_answer(
        query,
        retrieved_documents
    )

    latency = time.perf_counter() - start_time

    # Step 3: Extract retrieved chunk IDs
    retrieved_ids = [
        document["chunk_id"]
        for document in retrieved_documents
    ]

    # Step 4: Calculate retrieval metrics
    metrics = calculate_retrieval_metrics(
        retrieved_ids,
        gold_ids
    )

    # Step 5: Store the complete result
    result = {
        "case_id": case["case_id"],
        "query": query,
        "generated_answer": answer,

        "retrieved_documents": retrieved_documents,
        "retrieved_ids": retrieved_ids,
        "gold_evidence_ids": gold_ids,

        "retrieval_metrics": metrics,

        "execution": {
            "latency_seconds": latency,
            "token_usage": usage,
            "top_k": TOP_K
        }
    }

    return result

In [62]:
# Executing complete benchmark dataset

baseline_results = []

for i, case in enumerate(benchmark_cases, start=1):

    print(f"\nRunning case {i}/{len(benchmark_cases)}")

    result = run_baseline(case)

    baseline_results.append(result)

    print("Recall@K:", result["retrieval_metrics"]["recall_at_k"])
    print("Precision@K:", result["retrieval_metrics"]["precision_at_k"])
    print("MRR:", result["retrieval_metrics"]["mrr"])

    print("Latency:", round(
        result["execution"]["latency_seconds"], 2
    ), "seconds")


Running case 1/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 7.36 seconds

Running case 2/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 2.57 seconds

Running case 3/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 2.26 seconds

Running case 4/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 6.9 seconds

Running case 5/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 2.38 seconds

Running case 6/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 0.5
Latency: 5.26 seconds

Running case 7/18
Recall@K: 0.0
Precision@K: 0.0
MRR: 0.0
Latency: 16.78 seconds

Running case 8/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 0.3333333333333333
Latency: 2.93 seconds

Running case 9/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 2.15 seconds

Running case 10/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 4.43 seconds

Running case 11/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 0.5
Latency: 6.59 seconds

Running case 12/18
Recall@K: 1.0
Precision@K: 0.2
MRR: 1.0
Latency: 2.58 seconds

Running c

In [ ]:
# save baseline (benchmark) results

RESULTS_DIR = Path("../content/RAGspector/data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_FILE = RESULTS_DIR / "baseline_results_v1.jsonl"

with open(BASELINE_FILE, "w", encoding="utf-8") as file:

    for result in baseline_results:
        file.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

print("Baseline results saved to:", BASELINE_FILE)

Baseline results saved to: ..\content\RAGspector\data\results\baseline_results_v1.jsonl


In [65]:
# calculating aggregate baseline metrics

import numpy as np

recalls = [
    r["retrieval_metrics"]["recall_at_k"]
    for r in baseline_results
    if r["retrieval_metrics"]["recall_at_k"] is not None
]

precisions = [
    r["retrieval_metrics"]["precision_at_k"]
    for r in baseline_results
    if r["retrieval_metrics"]["precision_at_k"] is not None
]

mrr_scores = [
    r["retrieval_metrics"]["mrr"]
    for r in baseline_results
    if r["retrieval_metrics"]["mrr"] is not None
]

latencies = [
    r["execution"]["latency_seconds"]
    for r in baseline_results
]

print("========== BASELINE PERFORMANCE ==========")

print(f"Evaluated cases: {len(baseline_results)}")

print(f"Mean Recall@{TOP_K}: {np.mean(recalls):.4f}")
print(f"Mean Precision@{TOP_K}: {np.mean(precisions):.4f}")
print(f"Mean MRR: {np.mean(mrr_scores):.4f}")

print(f"Mean latency: {np.mean(latencies):.2f} seconds")
print(f"Median latency: {np.median(latencies):.2f} seconds")

========== BASELINE PERFORMANCE ==========
Evaluated cases: 18
Mean Recall@5: 0.8889
Mean Precision@5: 0.1778
Mean MRR: 0.7685
Mean latency: 5.06 seconds
Median latency: 4.59 seconds



---

**Generate Candidate Probes**

In [66]:
# CONFIG
import json
import random
import hashlib
import re
from collections import Counter
from pathlib import Path

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

BASELINE_TOP_K = TOP_K

# Supported interventions in the current local RAG implementation
SUPPORTED_CAPABILITIES = {
    "query_override",
    "top_k_override",
    "context_override"
}

# Initial retrieval configurations to test
RETRIEVAL_TOP_K_VALUES = [3, 8]

# One distractor per noise probe
NOISE_DOCUMENT_COUNT = 1

# Maximum query paraphrases generated per LLM request
PARAPHRASE_BATCH_SIZE = 6

In [70]:
# checking if baseline and benchmark results correspond to each other

assert len(benchmark_cases) == len(baseline_results), (
    "Benchmark and baseline result counts do not match."
)

benchmark_by_id = {
    case["case_id"]: case
    for case in benchmark_cases
}

baseline_by_id = {
    result["case_id"]: result
    for result in baseline_results
}

assert set(benchmark_by_id) == set(baseline_by_id), (
    "Benchmark and baseline case IDs do not match."
)

print("Benchmark cases:", len(benchmark_by_id))
print("Baseline results:", len(baseline_by_id))
print("Baseline top_k:", BASELINE_TOP_K)

Benchmark cases: 18
Baseline results: 18
Baseline top_k: 5


In [68]:
# Defining candidate probe structure

def create_probe(
    base_case_id,
    family,
    subtype,
    severity,
    intervention_level,
    expected_contract,
    rationale,
    modified_query=None,
    retrieval_options=None,
    context_intervention=None,
    estimated_cost=1.0,
    seed=RANDOM_SEED
):

    retrieval_options = retrieval_options or {}
    context_intervention = context_intervention or {}

    fingerprint_data = {
        "base_case_id": base_case_id,
        "family": family,
        "subtype": subtype,
        "modified_query": modified_query,
        "retrieval_options": retrieval_options,
        "context_intervention": context_intervention,
        "seed": seed
    }

    canonical_data = json.dumps(
        fingerprint_data,
        sort_keys=True,
        ensure_ascii=False,
        default=str
    )

    fingerprint = hashlib.sha256(
        canonical_data.encode("utf-8")
    ).hexdigest()

    probe = {
        "probe_id": "probe_" + fingerprint[:12],
        "base_case_id": base_case_id,
        "family": family,
        "subtype": subtype,
        "severity": severity,
        "intervention_level": intervention_level,
        "modified_query": modified_query,
        "retrieval_options": retrieval_options,
        "context_intervention": context_intervention,
        "expected_contract": expected_contract,
        "estimated_cost": estimated_cost,
        "seed": seed,
        "fingerprint": fingerprint,
        "rationale": rationale,
        "validity_checks": {}
    }

    return probe

*Query Robustness Probes (Query Paraphraser)*

In [69]:
def generate_paraphrases_batch(cases):

    input_questions = [
        {
            "case_id": case["case_id"],
            "question": case["query"]
        }
        for case in cases
    ]

    prompt = f"""
You are generating query paraphrases for a RAG reliability benchmark.

For each input question, generate exactly ONE paraphrase.

Rules:
1. Preserve the original meaning.
2. Preserve all named entities, dates and numerical values.
3. Preserve the requested answer type.
4. Do not introduce additional requirements.
5. Do not answer the question.
6. Do not add information that is not present in the original question.
7. Return the same case_id for each question.

Return ONLY a valid JSON array in this format:

[
    {{
        "case_id": "case_id",
        "paraphrased_question": "paraphrased question"
    }}
]

Input questions:

{json.dumps(input_questions, ensure_ascii=False)}
"""

    response = llm_client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        max_tokens=2000
    )

    raw_output = response.choices[0].message.content.strip()

    # Remove markdown code fences if present
    raw_output = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_output,
        flags=re.IGNORECASE
    ).strip()

    generated = json.loads(raw_output)

    if isinstance(generated, dict):
        generated = generated.get("paraphrases", [])

    return generated

In [71]:
# Run paraphrase generation in batches

# benchmark_cases : List

paraphrase_results = {}

for start in range(
    0,
    len(benchmark_cases),  #  <- pass only required number of cases here
    PARAPHRASE_BATCH_SIZE
):

    batch = benchmark_cases[
        start:start + PARAPHRASE_BATCH_SIZE
    ]

    print(
        f"Generating paraphrases for cases "
        f"{start + 1} to {start + len(batch)}"
    )

    generated = generate_paraphrases_batch(batch)

    for item in generated:

        case_id = item.get("case_id")
        paraphrase = item.get("paraphrased_question", "").strip()

        if case_id in benchmark_by_id and paraphrase:
            paraphrase_results[case_id] = paraphrase

print("\nGenerated paraphrases:", len(paraphrase_results))

Generating paraphrases for cases 1 to 6
Generating paraphrases for cases 7 to 12
Generating paraphrases for cases 13 to 18

Generated paraphrases: 18


In [72]:
# inspecting paraphrases and their original question

for case_id, paraphrase in list(paraphrase_results.items())[:5]:

    original = benchmark_by_id[case_id]["query"]

    print("\nCase:", case_id)
    print("Original:", original)
    print("Paraphrase:", paraphrase)
    print("-" * 80)


Case: wiki_d9430c285947
Original: Why can amides participate in hydrogen bonding with water?
Paraphrase: What enables amides to form hydrogen bonds with water?
--------------------------------------------------------------------------------

Case: wiki_a0ad3814aae9
Original: In what sequence do the cuffs inflate during external counterpulsation?
Paraphrase: What is the order in which the cuffs inflate during external counterpulsation?
--------------------------------------------------------------------------------

Case: wiki_3db9080a685c
Original: When did Varberg Fortress become Swedish territory?
Paraphrase: When did Varberg Fortress come under Swedish control?
--------------------------------------------------------------------------------

Case: wiki_f1b7c1e6ee5c
Original: What role did Ibrahim Tuqan play in Fadwa Tuqan's education and introduction to poetry?
Paraphrase: What part did Ibrahim Tuqan have in Fadwa Tuqan's education and her introduction to poetry?
------------------

In [73]:
# Create one query probe per accepted paraphrase

query_probes = []

for case_id, paraphrase in paraphrase_results.items():

    probe = create_probe(
        base_case_id=case_id,
        family="query_robustness",
        subtype="llm_paraphrase",
        severity="low",
        intervention_level="query",

        modified_query=paraphrase,

        expected_contract="invariant_answer",

        rationale=(
            "Tests whether a meaning-preserving change "
            "in question wording affects retrieval or answer quality."
        ),

        estimated_cost=1.0
    )

    query_probes.append(probe)

print("Query probes generated:", len(query_probes))

Query probes generated: 18



---

**Generate retrieval robustness probes**

In [74]:
""""
We will test two alternative configurations:
- top_k = 3: tests whether retrieving fewer chunks removes useful evidence.
- top_k = 8: tests whether retrieving more chunks improves evidence availability or introduces additional noise.
"""

retrieval_probes = []

for case in benchmark_cases:

    case_id = case["case_id"]

    for new_top_k in RETRIEVAL_TOP_K_VALUES:

        probe = create_probe(
            base_case_id=case_id,
            family="retrieval_robustness",
            subtype="top_k_variation",
            severity="low",
            intervention_level="retriever_config",

            retrieval_options={
                "top_k": new_top_k,
                "baseline_top_k": BASELINE_TOP_K
            },

            expected_contract="invariant_answer",

            rationale=(
                f"Tests the effect of changing top_k from "
                f"{BASELINE_TOP_K} to {new_top_k} on retrieval "
                "and downstream answer quality."
            ),

            estimated_cost=1.0
        )

        retrieval_probes.append(probe)

print("Retrieval probes generated:", len(retrieval_probes))

Retrieval probes generated: 36



---

**Generate noise robustness probes**

In [75]:
"""
This family tests whether irrelevant or weakly relevant passages influence the generated answer.

We will use the existing Wikipedia corpus to find candidate distractors. Specifically, we will:
1. Retrieve the top 20 chunks for a benchmark question.
2. Exclude the original top-five retrieved chunks.
3. Exclude gold evidence chunks.
4. Select one lower-ranked chunk as a distractor.
5. Append it to the original retrieved context.
"""


def get_candidate_distractor(case, baseline_result):

    query = case["query"]

    gold_ids = set(case.get("gold_evidence_ids", []))

    baseline_ids = set(baseline_result["retrieved_ids"])

    query_embedding = sen_trans.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=20,
        include=["documents", "metadatas", "distances"]
    )

    candidates = []

    for i, chunk_id in enumerate(results["ids"][0]):

        if chunk_id in gold_ids:
            continue

        if chunk_id in baseline_ids:
            continue

        document = results["documents"][0][i]
        metadata = results["metadatas"][0][i]

        if not document or len(document.strip()) < 200:
            continue

        candidates.append({
            "chunk_id": chunk_id,
            "text": document,
            "metadata": metadata,
            "distance": results["distances"][0][i]
        })

    # Prefer lower-ranked retrieved candidates
    if not candidates:
        return None

    return candidates[-1]

In [76]:
# Generating noise probes

noise_probes = []

for case in benchmark_cases:

    case_id = case["case_id"]
    baseline_result = baseline_by_id[case_id]

    distractor = get_candidate_distractor(
        case,
        baseline_result
    )

    if distractor is None:
        print("No distractor found for:", case_id)
        continue

    original_context = [
        {
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
            "metadata": doc["metadata"],
            "distance": doc["distance"]
        }
        for doc in baseline_result["retrieved_documents"]
    ]

    context_with_noise = original_context + [distractor]

    probe = create_probe(
        base_case_id=case_id,
        family="noise_robustness",
        subtype="single_distractor_insertion",
        severity="low",
        intervention_level="context",

        context_intervention={
            "type": "append_distractor",
            "documents": context_with_noise,
            "original_document_ids": baseline_result["retrieved_ids"],
            "inserted_document_ids": [distractor["chunk_id"]],
            "insertion_count": 1
        },

        expected_contract="invariant_answer",

        rationale=(
            "Tests whether an additional lower-ranked Wikipedia "
            "passage distracts the answer generator from the "
            "relevant evidence."
        ),

        estimated_cost=1.0
    )

    noise_probes.append(probe)

print("Noise probes generated:", len(noise_probes))

Noise probes generated: 18



---

**Generate context-order robustness probes**

In [79]:
"""
We will use the exact same documents retrieved during the baseline, but reverse their order.
Only the ordering changes. The number of documents, their IDs and their text remain identical.
"""

order_probes = []

for case in benchmark_cases:

    case_id = case["case_id"]
    baseline_result = baseline_by_id[case_id]

    original_context = [
        {
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
            "metadata": doc["metadata"],
            "distance": doc["distance"]
        }
        for doc in baseline_result["retrieved_documents"]
    ]

    if len(original_context) < 2:
        continue

    reversed_context = list(reversed(original_context))

    probe = create_probe(
        base_case_id=case_id,
        family="context_order_robustness",
        subtype="reverse_order",
        severity="low",
        intervention_level="context",

        context_intervention={
            "type": "reorder_documents",
            "documents": reversed_context,
            "original_document_ids": baseline_result["retrieved_ids"],
            "permutation": "reverse"
        },

        expected_contract="invariant_answer",

        rationale=(
            "Tests whether changing only the order of retrieved "
            "documents changes the correctness or faithfulness "
            "of the generated answer."
        ),

        estimated_cost=1.0
    )

    order_probes.append(probe)

print("Context-order probes generated:", len(order_probes))

Context-order probes generated: 18



---

**Conflict robustness probes**

In [80]:
# defining a function that accepts a manually verified fixture

def create_conflict_probe(case_id, fixture):

    required_fields = [
        "contradictory_text",
        "provenance",
        "verification_note"
    ]

    for field in required_fields:
        if not fixture.get(field):
            raise ValueError(f"Missing fixture field: {field}")

    if fixture.get("verified") is not True:
        raise ValueError(
            "Conflict fixture must be manually verified first."
        )

    case = benchmark_by_id[case_id]
    baseline_result = baseline_by_id[case_id]

    original_context = [
        {
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
            "metadata": doc["metadata"],
            "distance": doc["distance"]
        }
        for doc in baseline_result["retrieved_documents"]
    ]

    conflict_document = {
        "chunk_id": "synthetic_conflict_" + case_id,
        "text": fixture["contradictory_text"],
        "metadata": {
            "source": "controlled_synthetic_intervention",
            "provenance": fixture["provenance"],
            "source_authority": fixture.get("source_authority")
        },
        "distance": None
    }

    context_with_conflict = original_context + [conflict_document]

    expected_contract = (
        "resolve_by_source_authority"
        if fixture.get("source_authority")
        else "recognize_conflict_or_abstain"
    )

    return create_probe(
        base_case_id=case_id,
        family="conflict_robustness",
        subtype="verified_contradiction",
        severity="medium",
        intervention_level="context",

        context_intervention={
            "type": "insert_contradiction",
            "documents": context_with_conflict,
            "inserted_document_ids": [
                conflict_document["chunk_id"]
            ],
            "provenance": fixture["provenance"],
            "source_authority": fixture.get("source_authority"),
            "verification_note": fixture["verification_note"]
        },

        expected_contract=expected_contract,

        rationale=(
            "Tests whether the RAG system appropriately handles "
            "explicitly verified contradictory evidence."
        ),

        estimated_cost=1.0
    )


---

**Combine all generated probes**

In [81]:
candidate_probes = (
    query_probes
    + retrieval_probes
    + noise_probes
    + order_probes
)

print("Total candidates:", len(candidate_probes))

print("\nCandidates by family:")

family_counts = Counter(
    probe["family"]
    for probe in candidate_probes
)

for family, count in family_counts.items():
    print(f"{family}: {count}")

Total candidates: 90

Candidates by family:
query_robustness: 18
retrieval_robustness: 36
noise_robustness: 18
context_order_robustness: 18


**Note :**
Conflict probes will be additional candidates once verified fixtures are supplied.

**Validate the candidate probe**

In [82]:
MAX_QUERY_LENGTH = 2000
MAX_CONTEXT_CHARS = 30000

def validate_probe(probe, seen_fingerprints):

    errors = []

    required_fields = [
        "probe_id",
        "base_case_id",
        "family",
        "subtype",
        "severity",
        "intervention_level",
        "expected_contract",
        "fingerprint"
    ]

    # 1. Required fields
    for field in required_fields:
        if not probe.get(field):
            errors.append(f"Missing field: {field}")

    # 2. Base benchmark case
    if probe.get("base_case_id") not in benchmark_by_id:
        errors.append("Unknown base benchmark case")

    # 3. Expected contract
    if not probe.get("expected_contract"):
        errors.append("Missing expected behavioural contract")

    # 4. Duplicate detection
    if probe.get("fingerprint") in seen_fingerprints:
        errors.append("Duplicate candidate probe")

    # 5. Query intervention
    if probe["intervention_level"] == "query":

        modified_query = probe.get("modified_query")

        if not modified_query:
            errors.append("Modified query is empty")

        elif len(modified_query) > MAX_QUERY_LENGTH:
            errors.append("Modified query exceeds size limit")

        elif probe["base_case_id"] in benchmark_by_id:

            original_query = benchmark_by_id[
                probe["base_case_id"]
            ]["query"]

            if modified_query.strip() == original_query.strip():
                errors.append("Query was not modified")

    # 6. Retrieval configuration
    if probe["intervention_level"] == "retriever_config":

        options = probe.get("retrieval_options", {})
        new_top_k = options.get("top_k")

        if not isinstance(new_top_k, int) or new_top_k <= 0:
            errors.append("Invalid top_k configuration")

        elif new_top_k == BASELINE_TOP_K:
            errors.append("No change from baseline top_k")

        elif new_top_k > 20:
            errors.append("top_k exceeds configured limit")

    # 7. Context intervention
    if probe["intervention_level"] == "context":

        intervention = probe.get("context_intervention", {})
        documents = intervention.get("documents", [])

        if not documents:
            errors.append("Context intervention has no documents")

        else:

            total_chars = sum(
                len(doc.get("text", ""))
                for doc in documents
            )

            if total_chars > MAX_CONTEXT_CHARS:
                errors.append("Context exceeds configured size limit")

            if any(not doc.get("text") for doc in documents):
                errors.append("Context contains an empty document")

        # Noise-specific validation
        if probe["family"] == "noise_robustness":

            original_ids = set(
                intervention.get("original_document_ids", [])
            )

            inserted_ids = set(
                intervention.get("inserted_document_ids", [])
            )

            if not inserted_ids:
                errors.append("No distractor was inserted")

            if inserted_ids.intersection(original_ids):
                errors.append("Distractor duplicates original context")

        # Order-specific validation
        if probe["family"] == "context_order_robustness":

            original_ids = intervention.get(
                "original_document_ids", []
            )

            reordered_ids = [
                doc["chunk_id"]
                for doc in documents
            ]

            if reordered_ids != list(reversed(original_ids)):
                errors.append("Context permutation is incorrect")

    # 8. Capability validation
    required_capability = {
        "query": "query_override",
        "retriever_config": "top_k_override",
        "context": "context_override"
    }.get(probe["intervention_level"])

    if required_capability not in SUPPORTED_CAPABILITIES:
        errors.append("Unsupported intervention capability")

    # 9. Manual review requirements
    manual_review_required = probe["family"] in {
        "query_robustness",
        "noise_robustness",
        "conflict_robustness"
    }

    is_valid = len(errors) == 0

    probe["validity_checks"] = {
        "is_valid": is_valid,
        "errors": errors,
        "manual_review_required": manual_review_required,
        "manual_review_approved": not manual_review_required
    }

    if is_valid:
        seen_fingerprints.add(probe["fingerprint"])

    return probe

In [83]:
# Run validation

seen_fingerprints = set()

validated_probes = []

for probe in candidate_probes:

    validated = validate_probe(
        probe,
        seen_fingerprints
    )

    validated_probes.append(validated)

In [84]:
# Checking validation results

valid_probes = [
    p for p in validated_probes
    if p["validity_checks"]["is_valid"]
]

invalid_probes = [
    p for p in validated_probes
    if not p["validity_checks"]["is_valid"]
]

print("Total candidates:", len(validated_probes))
print("Valid candidates:", len(valid_probes))
print("Invalid candidates:", len(invalid_probes))

print("\nInvalid candidates:")

for probe in invalid_probes:
    print(
        probe["probe_id"],
        probe["validity_checks"]["errors"]
    )

Total candidates: 90
Valid candidates: 90
Invalid candidates: 0

Invalid candidates:


In [85]:
review_candidates = [
    p for p in valid_probes
    if p["validity_checks"]["manual_review_required"]
]

print("Candidates requiring review:", len(review_candidates))

for probe in review_candidates:

    case = benchmark_by_id[probe["base_case_id"]]

    print("\nProbe ID:", probe["probe_id"])
    print("Family:", probe["family"])
    print("Original question:", case["query"])

    if probe["family"] == "query_robustness":

        print(
            "Modified question:",
            probe["modified_query"]
        )

    elif probe["family"] == "noise_robustness":

        distractor = probe["context_intervention"]["documents"][-1]

        print("Distractor title:",
              distractor["metadata"].get("title"))

        print("Distractor text:",
              distractor["text"][:500])

    print("-" * 80)

Candidates requiring review: 36

Probe ID: probe_47b93d943a6d
Family: query_robustness
Original question: Why can amides participate in hydrogen bonding with water?
Modified question: What enables amides to form hydrogen bonds with water?
--------------------------------------------------------------------------------

Probe ID: probe_7d9ed9cb8663
Family: query_robustness
Original question: In what sequence do the cuffs inflate during external counterpulsation?
Modified question: What is the order in which the cuffs inflate during external counterpulsation?
--------------------------------------------------------------------------------

Probe ID: probe_0328c8c1ed5b
Family: query_robustness
Original question: When did Varberg Fortress become Swedish territory?
Modified question: When did Varberg Fortress come under Swedish control?
--------------------------------------------------------------------------------

Probe ID: probe_41f18518450c
Family: query_robustness
Original question: W

In [92]:
# approving all probes as of now

# Replace these with the IDs of candidates you actually reviewed.
approved_probe_ids = ["probe_47b93d943a6d", "probe_7d9ed9cb8663"]

for probe in valid_probes:

    if probe["probe_id"] in approved_probe_ids:

        probe["validity_checks"]["manual_review_approved"] = True


---

**Creating final candidate pool**

In [93]:
eligible_probes = [
    probe
    for probe in validated_probes
    if (
        probe["validity_checks"]["is_valid"]
        and
        probe["validity_checks"]["manual_review_approved"]
    )
]

print("Total generated candidates:", len(candidate_probes))
print("Valid candidates:", len(valid_probes))
print("Eligible for selection:", len(eligible_probes))

print("\nEligible candidates by family:")

eligible_counts = Counter(
    probe["family"]
    for probe in eligible_probes
)

for family, count in eligible_counts.items():
    print(f"{family}: {count}")

Total generated candidates: 90
Valid candidates: 90
Eligible for selection: 57

Eligible candidates by family:
query_robustness: 2
retrieval_robustness: 36
noise_robustness: 1
context_order_robustness: 18


**Saving the candidate pool**

In [94]:
CANDIDATE_DIR = Path("../content/RAGspector/data/candidates")
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

ALL_CANDIDATES_FILE = (
    CANDIDATE_DIR / "candidate_probes_v1.jsonl"
)

ELIGIBLE_CANDIDATES_FILE = (
    CANDIDATE_DIR / "eligible_probes_v1.jsonl"
)


def save_jsonl(records, filepath):

    with open(filepath, "w", encoding="utf-8") as file:

        for record in records:

            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )


save_jsonl(
    validated_probes,
    ALL_CANDIDATES_FILE
)

save_jsonl(
    eligible_probes,
    ELIGIBLE_CANDIDATES_FILE
)

print("All candidates saved:", ALL_CANDIDATES_FILE)
print("Eligible candidates saved:", ELIGIBLE_CANDIDATES_FILE)

All candidates saved: ..\content\RAGspector\data\candidates\candidate_probes_v1.jsonl
Eligible candidates saved: ..\content\RAGspector\data\candidates\eligible_probes_v1.jsonl
